# 阶段四项目实践（Colab 版）：RAG 与 Agent

> 适配你的学习路径：
> - P10: RAG 全链路（文档解析/Embedding/向量库/检索/生成）
> - P11: Advanced RAG（重排序/HyDE/多路召回）
> - P12: Agent 与工具调用（ReAct/Tool Use）

本 Notebook 强调“可直接在 Colab 跑起来”，每一节都给了最小可运行代码和可扩展任务。


## 0. 环境准备（Colab）


In [ ]:
# 如果你在 Colab 首次运行，请取消注释安装依赖
# !pip -q install -U sentence-transformers faiss-cpu rank-bm25 transformers accelerate

import json
import re
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np


## 1. 准备示例文档（可替换成你的真实资料）

为了让 Notebook 开箱即用，这里先构造一批“机器学习基础知识”文档。
后续你可替换成 PDF、Markdown、数据库文本。


In [ ]:
RAW_DOCS = [
    "Transformer 使用自注意力机制建模序列中任意位置之间的依赖关系。",
    "RAG 将外部知识检索与大语言模型生成结合，减少模型幻觉。",
    "向量检索通常使用余弦相似度或内积计算 query 与文档 embedding 的相关性。",
    "BM25 是经典的稀疏检索算法，适合关键词匹配场景。",
    "重排序器会对初筛候选文档进行更精细打分，提升 Top-k 结果质量。",
    "HyDE 会先让模型生成一个假设答案，再用假设答案去检索相关文档。",
    "Function Calling 是让模型按约定 JSON 结构输出工具调用参数。",
    "ReAct 将推理过程与行动过程交替执行，实现可解释的工具调用。",
    "向量数据库常见有 FAISS、Milvus、Weaviate、Qdrant。",
    "评估 RAG 常见指标包括 Recall@k、MRR、答案正确率与忠实性。",
]

print(f"文档数量: {len(RAW_DOCS)}")
print("示例:", RAW_DOCS[0])


---
## 2. P10：RAG 全链路（解析 → 向量化 → 检索 → 生成）


### 2.1 文本切分（Chunking）


In [ ]:
def simple_chunk(text: str, max_len: int = 30, overlap: int = 8) -> List[str]:
    text = text.strip()
    if len(text) <= max_len:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_len)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

chunks = []
for i, doc in enumerate(RAW_DOCS):
    for c in simple_chunk(doc, max_len=24, overlap=6):
        chunks.append({"doc_id": i, "text": c})

print("chunk 数:", len(chunks))
print(chunks[:3])


### 2.2 向量化 + FAISS 建库


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

chunk_texts = [x["text"] for x in chunks]
emb = embed_model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)
emb = np.asarray(emb, dtype=np.float32)

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

print("embedding shape:", emb.shape)
print("FAISS ntotal:", index.ntotal)


### 2.3 检索函数


In [ ]:
def dense_retrieve(query: str, k: int = 4):
    q = embed_model.encode([query], normalize_embeddings=True)
    q = np.asarray(q, dtype=np.float32)
    scores, idx = index.search(q, k)
    results = []
    for s, i in zip(scores[0], idx[0]):
        results.append({
            "score": float(s),
            "text": chunks[int(i)]["text"],
            "doc_id": chunks[int(i)]["doc_id"],
        })
    return results

query = "什么方法可以降低大模型幻觉？"
for r in dense_retrieve(query, k=3):
    print(r)


### 2.4 生成答案（RAG）


In [ ]:
from transformers import pipeline

gen = pipeline("text2text-generation", model="google/flan-t5-base")

def rag_answer(question: str, k: int = 4):
    ctxs = dense_retrieve(question, k=k)
    context_text = "\n".join([f"- {x['text']}" for x in ctxs])
    prompt = f"""
你是问答助手，请基于提供的上下文回答问题。
如果上下文没有答案，请明确说“不确定”。

问题：{question}
上下文：
{context_text}

请给出简洁答案：
""".strip()
    out = gen(prompt, max_new_tokens=128, do_sample=False)[0]["generated_text"]
    return out, ctxs

ans, ctx = rag_answer("RAG 为什么能减少幻觉？", k=4)
print("答案:\n", ans)
print("\n召回上下文:")
for c in ctx:
    print(c["text"])


### 2.5 P10 可交付要求
- 完成端到端流程（文档→检索→答案）
- 输出至少 10 个问题的结果表（问题/召回片段/答案）
- 给出失败案例分析（召回失败 or 生成偏离）


---
## 3. P11：Advanced RAG（重排序 / HyDE / 多路召回）


### 3.1 稀疏检索（BM25）与多路召回融合


In [ ]:
from rank_bm25 import BM25Okapi

def tokenize_zh_like(text: str):
    # 简化中文分词：按字切分（教学用途）
    return list(text)

bm25_corpus = [tokenize_zh_like(x["text"]) for x in chunks]
bm25 = BM25Okapi(bm25_corpus)

def bm25_retrieve(query: str, k: int = 4):
    q = tokenize_zh_like(query)
    scores = bm25.get_scores(q)
    idx = np.argsort(scores)[::-1][:k]
    return [{"score": float(scores[i]), "text": chunks[int(i)]["text"], "doc_id": chunks[int(i)]["doc_id"]} for i in idx]

def hybrid_retrieve(query: str, k_dense: int = 5, k_sparse: int = 5, k_final: int = 5):
    d = dense_retrieve(query, k_dense)
    b = bm25_retrieve(query, k_sparse)

    merged = {}
    for x in d:
        key = x["text"]
        merged.setdefault(key, {"text": key, "doc_id": x["doc_id"], "dense": 0.0, "bm25": 0.0})
        merged[key]["dense"] = max(merged[key]["dense"], x["score"])
    for x in b:
        key = x["text"]
        merged.setdefault(key, {"text": key, "doc_id": x["doc_id"], "dense": 0.0, "bm25": 0.0})
        merged[key]["bm25"] = max(merged[key]["bm25"], x["score"])

    # 简单归一化融合
    dense_vals = [v["dense"] for v in merged.values()]
    bm25_vals = [v["bm25"] for v in merged.values()]
    dmax = max(dense_vals) if dense_vals else 1.0
    bmax = max(bm25_vals) if bm25_vals else 1.0

    final = []
    for v in merged.values():
        score = 0.6 * (v["dense"] / (dmax + 1e-6)) + 0.4 * (v["bm25"] / (bmax + 1e-6))
        final.append({**v, "hybrid": score})

    final = sorted(final, key=lambda x: x["hybrid"], reverse=True)[:k_final]
    return final

print("Dense only:")
for x in dense_retrieve("什么是重排序", 3):
    print(x)

print("\nHybrid:")
for x in hybrid_retrieve("什么是重排序", 5, 5, 3):
    print(x)


### 3.2 HyDE 检索


In [ ]:
def hyde_retrieve(query: str, k: int = 4):
    hyde_prompt = f"请先写一段可能回答这个问题的简短段落：{query}"
    pseudo_answer = gen(hyde_prompt, max_new_tokens=64, do_sample=False)[0]["generated_text"]

    # 用“问题 + 假设答案”共同检索
    enhanced_query = query + "\n" + pseudo_answer
    return pseudo_answer, dense_retrieve(enhanced_query, k=k)

pseudo, hyde_res = hyde_retrieve("如何提高 RAG 召回质量？", k=3)
print("Pseudo answer:\n", pseudo)
print("\nHyDE 检索结果:")
for r in hyde_res:
    print(r)


### 3.3 简单重排序（Cross-Encoder 可选）

Colab 资源有限时，你可以先用“生成模型打分”模拟 rerank。


In [ ]:
def llm_rerank(query: str, candidates: List[Dict[str, Any]], top_k: int = 3):
    scored = []
    for c in candidates:
        prompt = f"""
请判断下面片段对回答问题是否有帮助，仅输出 0~1 之间的小数。
问题：{query}
片段：{c['text']}
分数：
""".strip()
        out = gen(prompt, max_new_tokens=8, do_sample=False)[0]["generated_text"]
        m = re.search(r"([01](?:\.\d+)?)", out)
        s = float(m.group(1)) if m else 0.0
        scored.append({**c, "rerank": s})
    return sorted(scored, key=lambda x: x["rerank"], reverse=True)[:top_k]

cands = hybrid_retrieve("RAG 的评估指标有哪些", 6, 6, 6)
reranked = llm_rerank("RAG 的评估指标有哪些", cands, top_k=3)
print(reranked)


### 3.4 P11 可交付要求
- 对比 Dense / BM25 / Hybrid / HyDE 至少 20 个问题
- 统计 Recall@k（手工标注相关文档即可）
- 给出“哪类问题适合哪种召回策略”的经验总结


---
## 4. P12：Agent 与工具调用（ReAct/Tool Use）


### 4.1 定义工具


In [ ]:
def tool_search_docs(query: str, k: int = 3):
    return hybrid_retrieve(query, k_dense=6, k_sparse=6, k_final=k)

def tool_calculator(expr: str):
    # 演示用途：仅允许数字与运算符
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr):
        return {"error": "表达式不安全"}
    try:
        return {"result": eval(expr, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

TOOLS = {
    "search_docs": tool_search_docs,
    "calculator": tool_calculator,
}


### 4.2 工具选择器（简化版 Function Calling）


In [ ]:
def decide_tool(user_query: str) -> Dict[str, Any]:
    prompt = f"""
你是工具路由器。可用工具：
1) search_docs(query: str, k: int)
2) calculator(expr: str)

根据用户问题选择一个工具，并只输出 JSON：
{{"tool_name": "...", "arguments": {{...}}}}

用户问题：{user_query}
""".strip()
    out = gen(prompt, max_new_tokens=128, do_sample=False)[0]["generated_text"]

    # 尝试提取 JSON
    m = re.search(r"\{[\s\S]*\}", out)
    if not m:
        # fallback 规则
        if re.search(r"[0-9][0-9+\-*/(). ]+[0-9]", user_query):
            return {"tool_name": "calculator", "arguments": {"expr": user_query}}
        return {"tool_name": "search_docs", "arguments": {"query": user_query, "k": 3}}

    try:
        return json.loads(m.group(0))
    except Exception:
        if re.search(r"[0-9][0-9+\-*/(). ]+[0-9]", user_query):
            return {"tool_name": "calculator", "arguments": {"expr": user_query}}
        return {"tool_name": "search_docs", "arguments": {"query": user_query, "k": 3}}


### 4.3 Agent 主循环（Think → Act → Observe → Answer）


In [ ]:
def agent_answer(user_query: str):
    decision = decide_tool(user_query)
    tool_name = decision.get("tool_name")
    arguments = decision.get("arguments", {})

    if tool_name not in TOOLS:
        tool_name = "search_docs"
        arguments = {"query": user_query, "k": 3}

    try:
        observation = TOOLS[tool_name](**arguments)
    except Exception as e:
        observation = {"error": str(e)}

    final_prompt = f"""
你是一个可靠助手，请基于工具结果回答用户问题。
用户问题：{user_query}
工具：{tool_name}
工具参数：{arguments}
工具结果：{observation}

请给出最终答案（中文，简洁准确）：
""".strip()

    final = gen(final_prompt, max_new_tokens=128, do_sample=False)[0]["generated_text"]
    return {
        "decision": decision,
        "observation": observation,
        "final_answer": final,
    }

case1 = agent_answer("RAG 常用评估指标有哪些？")
case2 = agent_answer("请计算 (23*7+9)/4")

print("Case1:\n", case1)
print("\nCase2:\n", case2)


### 4.4 P12 可交付要求
- 至少 3 个工具（检索/计算/外部 API）
- 至少 30 条测试样例，统计工具选择准确率
- 记录失败样例：JSON 解析失败、工具参数错误、结果幻觉


---
## 5. 阶段四统一验收清单

- [ ] P10：可运行的 RAG 基线 + 结果表
- [ ] P11：至少两种高级检索策略对比 + 指标
- [ ] P12：完整工具调用闭环 + 错误处理
- [ ] 每个项目写“结论与下一步优化”

你完成这份 Notebook 后，阶段四的核心工程能力就具备了：
**可检索、可对比、可调用工具、可评估**。
